# Revision experiments — closing the weak points before submission

Run top to bottom. Nothing from Phase 1 or Phase 2 is retrained; everything here
is additive and saves into `results/revision/`.

| Section | Weak point it closes | Trains? | Time |
|---|---|---|---|
| 1 | **Take-level confound** — sibling views of a test take sit in training | yes | ~1.5 h |
| 2 | Single-seed ladder rungs | no (CPU) | ~20 min |
| 3 | Closed-set assumption only | no (CPU) | ~5 min |
| 4 | "Identity is just bone length" | yes | ~1.5 h |
| 5 | Single backbone (ST-GCN only) | yes | ~2 h |
| 6 | Assemble the revision tables | no | seconds |

Section 1 is the one that must not be skipped: it protects the headline numbers.
Sections 2 and 3 are nearly free. Section 5 is the most expensive and can be
deferred to revision if your deadline is tight.

## 0 · Setup (definitions and cache — no training)

## 0 · Setup — mount Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 0 · Setup — paths and constants

In [ ]:
import os, sys, re, json, time, math, pickle, random
import numpy as np

SEED = 1
random.seed(SEED); np.random.seed(SEED)

IN_COLAB  = 'google.colab' in sys.modules
IN_KAGGLE = os.path.exists('/kaggle')

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    PROJECT = '/content/drive/MyDrive/PhD_Q1'
    SCRATCH = '/content/scratch'
elif IN_KAGGLE:
    PROJECT = '/kaggle/working/PhD_Q1'
    SCRATCH = '/kaggle/working/scratch'
else:
    PROJECT = os.path.abspath('./PhD_Q1')
    SCRATCH = os.path.abspath('./scratch')

DATA_DIR = os.path.join(PROJECT, 'data')
CKPT_DIR = os.path.join(PROJECT, 'checkpoints')
RES_DIR  = os.path.join(PROJECT, 'results')
for d in (DATA_DIR, CKPT_DIR, RES_DIR, SCRATCH):
    os.makedirs(d, exist_ok=True)

PKL_PATH  = os.path.join(DATA_DIR, 'ntu60_3danno.pkl')
CACHE_X   = os.path.join(DATA_DIR, 'ntu60_T64_x.npy')
CACHE_M   = os.path.join(DATA_DIR, 'ntu60_T64_meta.npz')

# geometry / tensor shapes
NUM_NODE, MAX_PERSON, T_LEN, IN_CH = 25, 2, 64, 3

print('project :', PROJECT)
print('pkl     :', PKL_PATH, '->', os.path.exists(PKL_PATH))

## 0 · Setup — torch and device

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(SEED)
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('torch', torch.__version__, '| device:', DEVICE)
if DEVICE == 'cuda':
    print(torch.cuda.get_device_name(0))
else:
    print('WARNING: no GPU detected. Colab: Runtime > Change runtime type > GPU.')

## 1 · Load the NTU annotations

In [ ]:
t0 = time.time()
with open(PKL_PATH, 'rb') as f:
    D = pickle.load(f)
print('loaded in %.1f s' % (time.time() - t0))

anns  = D['annotations']
names = [a['frame_dir'] for a in anns]

setup   = np.array([int(n[1:4])   for n in names], dtype=np.int16)
camera  = np.array([int(n[5:8])   for n in names], dtype=np.int16)
subject = np.array([int(n[9:12])  for n in names], dtype=np.int16)
action  = np.array([a['label']    for a in anns],  dtype=np.int16)

print('samples  :', len(anns))
print('subjects :', len(np.unique(subject)))
print('cameras  :', np.unique(camera))
print('actions  :', len(np.unique(action)))
print('example  :', names[0])
print('keys     :', list(anns[0].keys()))
print('keypoint :', np.asarray(anns[0]['keypoint']).shape, ' (M, T, V, C)')

assert len(np.unique(subject)) == 40, 'expected 40 subjects'
print('\nOK — identity labels present. Proceed.')

## 2 · Preprocessing functions

In [ ]:
def _unit(v):
    n = np.linalg.norm(v)
    return v if n < 1e-8 else v / n

def _rotation_matrix(axis, theta):
    axis = _unit(np.asarray(axis, dtype=np.float64))
    a = math.cos(theta / 2.0)
    b, c, d = -axis * math.sin(theta / 2.0)
    aa, bb, cc, dd = a*a, b*b, c*c, d*d
    bc, ad, ac, ab, bd, cd = b*c, a*d, a*c, a*b, b*d, c*d
    return np.array([[aa+bb-cc-dd, 2*(bc+ad),   2*(bd-ac)],
                     [2*(bc-ad),   aa+cc-bb-dd, 2*(cd+ab)],
                     [2*(bd+ac),   2*(cd-ab),   aa+dd-bb-cc]], dtype=np.float32)

def _angle_between(v1, v2):
    return np.arccos(np.clip(np.dot(_unit(v1), _unit(v2)), -1.0, 1.0))

def _align(skel, ja, jb, target):
    # rotate the whole sequence so that joint ja -> jb points along `target`
    vec = skel[0, 0, jb] - skel[0, 0, ja]
    if np.linalg.norm(vec) < 1e-8:
        return skel
    axis, angle = np.cross(vec, target), _angle_between(vec, target)
    if np.linalg.norm(axis) < 1e-8 or angle < 1e-8:
        return skel
    return (skel @ _rotation_matrix(axis, angle).T).astype(np.float32)

def pre_normalize(kp):
    kp = np.asarray(kp, dtype=np.float32).copy()
    valid = kp.sum(axis=(0, 2, 3)) != 0
    if valid.any():
        kp = kp[:, valid]
    if kp.shape[1] == 0:
        return np.zeros((1, 1, NUM_NODE, 3), dtype=np.float32)
    mask = kp.sum(axis=-1) != 0
    kp[mask] = kp[mask] - kp[0, 0, 1].copy()          # centre on spine
    kp = _align(kp, 0, 1, np.array([0, 0, 1], np.float32))   # spine     -> z
    kp = _align(kp, 8, 4, np.array([1, 0, 0], np.float32))   # shoulders -> x
    return kp

def resample(kp, t_len=T_LEN):
    M, T, V, Ch = kp.shape
    out = kp[:, np.linspace(0, T - 1, t_len).round().astype(np.int64)]
    if M < MAX_PERSON:
        out = np.concatenate(
            [out, np.zeros((MAX_PERSON - M, t_len, V, Ch), out.dtype)], axis=0)
    return out[:MAX_PERSON]

# quick self-test on synthetic data
_t = np.random.RandomState(0).normal(size=(1, 90, 25, 3)).astype(np.float32)
assert resample(pre_normalize(_t)).shape == (MAX_PERSON, T_LEN, NUM_NODE, 3)
print('preprocessing OK')

## 2 · Build or load the cache

In [ ]:
if os.path.exists(CACHE_X) and os.path.exists(CACHE_M):
    print('cache found, skipping preprocessing')
else:
    N = len(anns)
    X = np.zeros((N, MAX_PERSON, T_LEN, NUM_NODE, IN_CH), dtype=np.float16)
    t0 = time.time()
    for i, a in enumerate(anns):
        X[i] = resample(pre_normalize(np.asarray(a['keypoint']))).astype(np.float16)
        if (i + 1) % 5000 == 0:
            el = time.time() - t0
            print('  %6d / %d   %.0f s elapsed, ~%.0f s left'
                  % (i + 1, N, el, el / (i + 1) * (N - i - 1)))
    np.save(CACHE_X, X)
    np.savez(CACHE_M, setup=setup, camera=camera, subject=subject,
             action=action, names=np.array(names))
    print('cached in %.0f s -> %.2f GB' % (time.time() - t0, X.nbytes / 1e9))
    del X

## 2 · Copy the cache to local disk

In [ ]:
# copy to local disk first: Drive I/O is slow for repeated reads
LOCAL_X = os.path.join(SCRATCH, 'x.npy')
if not os.path.exists(LOCAL_X):
    import shutil; shutil.copy(CACHE_X, LOCAL_X)

X = np.load(LOCAL_X)                       # (N, 2, 64, 25, 3) float16
_m = np.load(CACHE_M, allow_pickle=True)
setup, camera, subject, action = _m['setup'], _m['camera'], _m['subject'], _m['action']
names = list(_m['names'])

SUBJ_IDS   = sorted(np.unique(subject).tolist())
SUBJ_LABEL = np.array([SUBJ_IDS.index(s) for s in subject], dtype=np.int64)  # 0..39
ACT_LABEL  = action.astype(np.int64)                                         # 0..59

print('X', X.shape, X.dtype, '%.2f GB' % (X.nbytes / 1e9))
print('action classes', ACT_LABEL.max() + 1, '| identity classes', SUBJ_LABEL.max() + 1)

## 3 · Evaluation protocols (splits)

In [ ]:
XSUB_TRAIN = [1,2,4,5,8,9,13,14,15,16,17,18,19,25,27,28,31,34,35,38]

SPLITS = {}
SPLITS['xsub'] = (np.where(np.isin(subject, XSUB_TRAIN))[0],
                  np.where(~np.isin(subject, XSUB_TRAIN))[0])

rs, tr, te = np.random.RandomState(SEED), [], []
for s in SUBJ_IDS:                                   # stratified 80/20 per person
    idx = np.where(subject == s)[0]; rs.shuffle(idx)
    k = int(round(0.8 * len(idx)))
    tr += idx[:k].tolist(); te += idx[k:].tolist()
SPLITS['reid_rand'] = (np.array(sorted(tr)), np.array(sorted(te)))

SPLITS['reid_xview'] = (np.where(np.isin(camera, [1, 2]))[0],
                        np.where(camera == 3)[0])

for k, (a, b) in SPLITS.items():
    print('%-11s train %6d  test %6d  | subjects %2d / %2d'
          % (k, len(a), len(b), len(set(subject[a])), len(set(subject[b]))))

assert len(set(subject[SPLITS['xsub'][0]]) & set(subject[SPLITS['xsub'][1]])) == 0
assert len(set(subject[SPLITS['reid_xview'][0]]) &
           set(subject[SPLITS['reid_xview'][1]])) == 40

## 4 · ST-GCN definition

In [ ]:
_INWARD_1B = [(1,2),(2,21),(3,21),(4,3),(5,21),(6,5),(7,6),(8,7),(9,21),
              (10,9),(11,10),(12,11),(13,1),(14,13),(15,14),(16,15),
              (17,1),(18,17),(19,18),(20,19),(22,23),(23,8),(24,25),(25,12)]
INWARD    = [(i-1, j-1) for i, j in _INWARD_1B]
OUTWARD   = [(j, i) for i, j in INWARD]
SELF_LINK = [(i, i) for i in range(NUM_NODE)]

def _edge2mat(link, n):
    A = np.zeros((n, n), dtype=np.float32)
    for i, j in link:
        A[j, i] = 1.0
    return A

def _norm_digraph(A):
    Dl, Dn = A.sum(0), np.zeros_like(A)
    for i in range(A.shape[0]):
        if Dl[i] > 0:
            Dn[i, i] = Dl[i] ** -1.0
    return A @ Dn

def build_adjacency():
    return np.stack([_edge2mat(SELF_LINK, NUM_NODE),
                     _norm_digraph(_edge2mat(INWARD, NUM_NODE)),
                     _norm_digraph(_edge2mat(OUTWARD, NUM_NODE))]).astype(np.float32)


class SpatialGC(nn.Module):
    def __init__(self, in_c, out_c, A):
        super().__init__()
        self.K = A.shape[0]
        self.conv = nn.Conv2d(in_c, out_c * self.K, 1)
        self.register_buffer('A', torch.from_numpy(A.copy()))

    def forward(self, x):
        N, _, T, V = x.shape
        x = self.conv(x).view(N, self.K, -1, T, V)
        return torch.einsum('nkctv,kvw->nctw', x, self.A.to(x.dtype)).contiguous()


class STGCNBlock(nn.Module):
    def __init__(self, in_c, out_c, A, stride=1, residual=True):
        super().__init__()
        self.gcn = SpatialGC(in_c, out_c, A)
        self.bn1 = nn.BatchNorm2d(out_c)
        self.tcn = nn.Conv2d(out_c, out_c, (9, 1), (stride, 1), (4, 0))
        self.bn2 = nn.BatchNorm2d(out_c)
        self.drop = nn.Dropout(0.1)
        if not residual:
            self.res = None
        elif in_c == out_c and stride == 1:
            self.res = nn.Identity()
        else:
            self.res = nn.Sequential(nn.Conv2d(in_c, out_c, 1, (stride, 1)),
                                     nn.BatchNorm2d(out_c))

    def forward(self, x):
        r = 0 if self.res is None else self.res(x)
        y = F.relu(self.bn1(self.gcn(x)))
        y = self.drop(self.bn2(self.tcn(y)))
        return F.relu(y + r)


class STGCN(nn.Module):
    def __init__(self, num_class, in_channels=IN_CH, num_person=MAX_PERSON, dropout=0.3):
        super().__init__()
        A = build_adjacency()
        self.data_bn = nn.BatchNorm1d(num_person * in_channels * NUM_NODE)
        cfg = [(in_channels, 64, 1, False), (64, 64, 1, True), (64, 64, 1, True),
               (64, 64, 1, True), (64, 128, 2, True), (128, 128, 1, True),
               (128, 128, 1, True), (128, 256, 2, True), (256, 256, 1, True),
               (256, 256, 1, True)]
        self.layers = nn.ModuleList([STGCNBlock(i, o, A, s, r) for i, o, s, r in cfg])
        self.drop, self.fc, self.feat_dim = nn.Dropout(dropout), nn.Linear(256, num_class), 256

    def extract(self, x):                      # x: N, C, T, V, M
        N, C, T, V, M = x.shape
        x = x.permute(0, 4, 3, 1, 2).contiguous().view(N, M * V * C, T)
        x = self.data_bn(x)
        x = x.view(N, M, V, C, T).permute(0, 1, 3, 4, 2).contiguous().view(N * M, C, T, V)
        for l in self.layers:
            x = l(x)
        return F.adaptive_avg_pool2d(x, 1).view(N, M, -1).mean(1)

    def forward(self, x, return_feat=False):
        f = self.extract(x)
        out = self.fc(self.drop(f))
        return (out, f) if return_feat else out


_m = STGCN(60)
_o, _f = _m(torch.randn(2, IN_CH, T_LEN, NUM_NODE, MAX_PERSON), return_feat=True)
print('logits', tuple(_o.shape), '| features', tuple(_f.shape))
print('parameters: %.2f M' % (sum(p.numel() for p in _m.parameters()) / 1e6))
del _m, _o, _f

## 5 · Training engine

In [ ]:
def _euler(theta):
    cx, cy, cz = np.cos(theta); sx, sy, sz = np.sin(theta)
    Rx = np.array([[1,0,0],[0,cx,-sx],[0,sx,cx]])
    Ry = np.array([[cy,0,sy],[0,1,0],[-sy,0,cy]])
    Rz = np.array([[cz,-sz,0],[sz,cz,0],[0,0,1]])
    return (Rz @ Ry @ Rx).astype(np.float32)

def augment(x, rot=0.3, scale=0.1):
    R = _euler(np.random.uniform(-rot, rot, 3))
    s = 1.0 + np.random.uniform(-scale, scale)
    mask = (x.sum(-1) != 0)[..., None]
    return np.where(mask, (x @ R.T) * s, 0.0).astype(np.float32)


class SkeletonDS(torch.utils.data.Dataset):
    def __init__(self, X, y, idx, train=False):
        self.X, self.y, self.idx, self.train = X, y, idx, train
    def __len__(self):
        return len(self.idx)
    def __getitem__(self, i):
        j = self.idx[i]
        x = self.X[j].astype(np.float32)
        if self.train:
            x = augment(x)
        x = np.ascontiguousarray(np.transpose(x, (3, 1, 2, 0)))   # C,T,V,M
        return torch.from_numpy(x), int(self.y[j])


def _load_ckpt(p):
    try:
        return torch.load(p, map_location='cpu', weights_only=False)
    except TypeError:
        return torch.load(p, map_location='cpu')


@torch.no_grad()
def evaluate(model, loader):
    model.eval(); correct = total = 0
    for xb, yb in loader:
        xb, yb = xb.to(DEVICE, non_blocking=True), yb.to(DEVICE, non_blocking=True)
        with torch.autocast('cuda', enabled=(DEVICE == 'cuda')):
            pred = model(xb).argmax(1)
        correct += (pred == yb).sum().item(); total += yb.numel()
    return 100.0 * correct / total


def run_experiment(tag, labels, split, num_classes, epochs=60, lr=0.1, bs=64,
                   wd=5e-4, warmup=5, workers=2, init_from=None, freeze=False):
    tr_idx, te_idx = SPLITS[split]
    tl = torch.utils.data.DataLoader(SkeletonDS(X, labels, tr_idx, True), bs, shuffle=True,
                                     num_workers=workers, pin_memory=(DEVICE == 'cuda'),
                                     drop_last=True, persistent_workers=workers > 0)
    vl = torch.utils.data.DataLoader(SkeletonDS(X, labels, te_idx, False), bs * 2,
                                     shuffle=False, num_workers=workers,
                                     pin_memory=(DEVICE == 'cuda'))

    model = STGCN(num_classes).to(DEVICE)
    if init_from:
        sd = _load_ckpt(init_from)['model']
        sd = {k: v for k, v in sd.items() if not k.startswith('fc.')}
        print('  init from', os.path.basename(init_from),
              model.load_state_dict(sd, strict=False).missing_keys[:2], '...')
    if freeze:
        for n_, p_ in model.named_parameters():
            if not n_.startswith('fc.'):
                p_.requires_grad = False

    params = [p for p in model.parameters() if p.requires_grad]
    opt = torch.optim.SGD(params, lr=lr, momentum=0.9, nesterov=True, weight_decay=wd)
    crit = nn.CrossEntropyLoss(label_smoothing=0.1)
    scaler = torch.amp.GradScaler('cuda', enabled=(DEVICE == 'cuda'))

    last = os.path.join(CKPT_DIR, tag + '_last.pt')
    best_p = os.path.join(CKPT_DIR, tag + '_best.pt')
    start, best, hist = 0, 0.0, []
    if os.path.exists(last):
        ck = _load_ckpt(last)
        model.load_state_dict(ck['model']); opt.load_state_dict(ck['opt'])
        start, best, hist = ck['epoch'] + 1, ck['best'], ck['hist']
        print('  resumed at epoch %d (best %.2f%%)' % (start, best))

    def set_lr(ep):
        f = (ep + 1) / warmup if ep < warmup else \
            0.5 * (1 + math.cos(math.pi * (ep - warmup) / max(1, epochs - warmup)))
        for g in opt.param_groups:
            g['lr'] = lr * f
        return lr * f

    print('[%s] split=%s  classes=%d  train=%d  test=%d'
          % (tag, split, num_classes, len(tr_idx), len(te_idx)))
    for ep in range(start, epochs):
        cur_lr = set_lr(ep); model.train(); t0 = time.time(); run = 0.0
        for xb, yb in tl:
            xb, yb = xb.to(DEVICE, non_blocking=True), yb.to(DEVICE, non_blocking=True)
            opt.zero_grad(set_to_none=True)
            with torch.autocast('cuda', enabled=(DEVICE == 'cuda')):
                loss = crit(model(xb), yb)
            scaler.scale(loss).backward(); scaler.step(opt); scaler.update()
            run += loss.item()
        acc = evaluate(model, vl)
        hist.append({'epoch': ep, 'loss': run / max(1, len(tl)), 'acc': acc, 'lr': cur_lr})
        torch.save({'epoch': ep, 'model': model.state_dict(), 'opt': opt.state_dict(),
                    'best': max(best, acc), 'hist': hist}, last)
        if acc > best:
            best = acc; torch.save({'model': model.state_dict(), 'acc': acc}, best_p)
        print('  ep %3d/%d  loss %.3f  acc %6.2f%%  best %6.2f%%  lr %.4f  (%.0fs)'
              % (ep + 1, epochs, hist[-1]['loss'], acc, best, cur_lr, time.time() - t0))

    json.dump(hist, open(os.path.join(RES_DIR, tag + '_history.json'), 'w'), indent=1)
    print('[%s] BEST %.2f%%' % (tag, best))
    return {'tag': tag, 'split': split, 'classes': num_classes,
            'best_acc': best, 'ckpt': best_p}

RESULTS = {}
RES_JSON = os.path.join(RES_DIR, 'phase1_results.json')
if os.path.exists(RES_JSON):
    RESULTS = json.load(open(RES_JSON))
    print('loaded previous results:', list(RESULTS))

def save_results():
    json.dump(RESULTS, open(RES_JSON, 'w'), indent=2)

## 0b · Load Phase 1 and Phase 2 results

In [ ]:
import json, os, glob
REV_DIR = os.path.join(RES_DIR, 'revision')
os.makedirs(REV_DIR, exist_ok=True)
REV = json.load(open(os.path.join(REV_DIR, 'revision.json'))) \
      if os.path.exists(os.path.join(REV_DIR, 'revision.json')) else {}

RESULTS = json.load(open(os.path.join(RES_DIR, 'phase1_results.json')))
P2 = json.load(open(os.path.join(RES_DIR, 'phase2', 'p2_summary.json')))
print('Phase 1:', {k: round(v['best_acc'], 2) for k, v in RESULTS.items()})
print('Phase 2 lambdas:', sorted([k for k in P2 if k[0].isdigit()], key=float))

def save_rev():
    json.dump(REV, open(os.path.join(REV_DIR, 'revision.json'), 'w'), indent=2)

from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.neural_network import MLPClassifier

## 1 · Take-level confound — the priority experiment

NTU films every performance with three cameras **at the same time**. In the
random split and in the cameras-1&2-to-3 split, other views of a test take are
usually present in training, so part of the measured leakage may be matching the
same performance across views rather than recognising the person.

Two new protocols fix this:

- **take-grouped**: all three views of a take (setup, performer, replication,
  action) are kept on the same side of the split, so no test take is ever seen
  from another angle during training;
- **cross-setup**: train on even-numbered setups, test on odd — a different
  recording session as well as a different view.

If re-identification stays well above chance under both, the headline claim is
safe. Expect a drop from 78.86%; a fall to near chance would mean the original
figures were largely take matching and the paper needs rewriting.

In [ ]:
# replication index is the R field of the NTU file name (S###C###P###R###A###)
replication = np.array([int(n[13:16]) for n in names], dtype=np.int16)

take_id = (setup.astype(np.int64) * 10**9 + subject.astype(np.int64) * 10**6
           + replication.astype(np.int64) * 10**3 + action.astype(np.int64))
uniq_takes = np.unique(take_id)
print('distinct takes: %d  (%.2f clips per take)'
      % (len(uniq_takes), len(take_id) / len(uniq_takes)))

# take-grouped 80/20, stratified by subject so every identity appears both sides
rs = np.random.RandomState(SEED)
tr_t, te_t = [], []
for s in SUBJ_IDS:
    t_s = np.unique(take_id[subject == s])
    rs.shuffle(t_s)
    k = int(round(0.8 * len(t_s)))
    tr_t += t_s[:k].tolist(); te_t += t_s[k:].tolist()
tr_t, te_t = set(tr_t), set(te_t)
SPLITS['reid_take'] = (np.where(np.isin(take_id, list(tr_t)))[0],
                       np.where(np.isin(take_id, list(te_t)))[0])

# cross-setup: even setups train, odd test
SPLITS['reid_xsetup'] = (np.where(setup % 2 == 0)[0], np.where(setup % 2 == 1)[0])

for k in ['reid_rand', 'reid_take', 'reid_xsetup']:
    a, b = SPLITS[k]
    shared = len(set(take_id[a]) & set(take_id[b]))
    print('%-12s train %6d  test %6d  subjects %2d/%2d  shared takes %d'
          % (k, len(a), len(b), len(set(subject[a])), len(set(subject[b])), shared))

assert len(set(take_id[SPLITS['reid_take'][0]]) &
           set(take_id[SPLITS['reid_take'][1]])) == 0, 'take leakage remains'
print('\ntake-grouped split verified: no take appears on both sides')

In [ ]:
for tag, sp in [('B_reid_take', 'reid_take'), ('B_reid_xsetup', 'reid_xsetup')]:
    if tag in REV:
        print('skipping %s (done, %.2f%%)' % (tag, REV[tag]['best_acc'])); continue
    REV[tag] = run_experiment(tag=tag, labels=SUBJ_LABEL, split=sp,
                              num_classes=40, epochs=80, lr=0.1, bs=64)
    save_rev()

print('\n--- re-identification from scratch, chance 2.50%% ---')
print('%-26s %8s %9s' % ('protocol', 'top-1', 'x chance'))
print('%-26s %7.2f%% %8.1fx' % ('random (original)', 78.86, 78.86 / 2.5))
print('%-26s %7.2f%% %8.1fx' % ('cameras 1,2 -> 3 (original)', 56.26, 56.26 / 2.5))
for tag, lab in [('B_reid_take', 'take-grouped'), ('B_reid_xsetup', 'cross-setup')]:
    a = REV[tag]['best_acc']
    print('%-26s %7.2f%% %8.1fx' % (lab, a, a / 2.5))

## 2 · Three-seed ladder rungs

The R2 and R3 figures in Table 4 rest on one action model. You already have two
further action models from the seed runs, so the probes can be repeated on those
at no training cost. This turns three single-seed numbers into mean ± SD.

The convex probe (R2) is deterministic given the features, so its variation
comes entirely from the backbone seed; the MLP probe (R3) also varies with its
own initialisation, which we fix.

In [ ]:
@torch.no_grad()
def plain_features(ckpt, idx, bs=256):
    m = STGCN(60).to(DEVICE)
    sd = _load_ckpt(ckpt)['model']
    sd = {k: v for k, v in sd.items() if not k.startswith('fc.')}
    m.load_state_dict(sd, strict=False); m.eval()
    ld = torch.utils.data.DataLoader(SkeletonDS(X, SUBJ_LABEL, idx, False), bs,
                                     shuffle=False, num_workers=4)
    o = []
    for xb, _ in ld:
        with torch.autocast('cuda', enabled=(DEVICE == 'cuda')):
            f = m.extract(xb.to(DEVICE))
        if f.ndim > 2:
            f = f.mean(dim=tuple(range(2, f.ndim)))
        o.append(f.float().cpu().numpy())
    del m
    if DEVICE == 'cuda':
        torch.cuda.empty_cache()
    return np.concatenate(o, 0)


def ladder_probes(ckpt, split='reid_rand'):
    tr, te = SPLITS[split]
    tr, te = np.sort(tr), np.sort(te)
    Ftr, Fte = plain_features(ckpt, tr), plain_features(ckpt, te)
    ytr, yte = np.asarray(SUBJ_LABEL)[tr], np.asarray(SUBJ_LABEL)[te]
    sc = StandardScaler().fit(Ftr)
    Ztr, Zte = sc.transform(Ftr), sc.transform(Fte)
    return {
        'linear': 100 * LogisticRegression(max_iter=3000).fit(Ztr, ytr).score(Zte, yte),
        'mlp': 100 * MLPClassifier(hidden_layer_sizes=(512, 256), max_iter=300,
                                   early_stopping=True, n_iter_no_change=15,
                                   random_state=1).fit(Ztr, ytr).score(Zte, yte)}


SEED_CKPTS = {'seed0': RESULTS['A_action_xsub']['ckpt']}
for s in ['s2', 's3']:
    f = os.path.join(CKPT_DIR, 'A_action_xsub_%s_best.pt' % s)
    if os.path.exists(f):
        SEED_CKPTS['seed_' + s] = f

if 'ladder_seeds' not in REV:
    REV['ladder_seeds'] = {}
for name, ck in SEED_CKPTS.items():
    if name in REV['ladder_seeds']:
        continue
    REV['ladder_seeds'][name] = ladder_probes(ck)
    print('%-9s R2 %.2f%%  R3 %.2f%%' % (name, REV['ladder_seeds'][name]['linear'],
                                          REV['ladder_seeds'][name]['mlp']))
    save_rev()

for fld, lab in [('linear', 'R2 linear probe'), ('mlp', 'R3 MLP probe')]:
    v = np.array([r[fld] for r in REV['ladder_seeds'].values()])
    print('%-16s %.2f +/- %.2f  (n=%d)' % (lab, v.mean(), v.std(ddof=1), len(v)))

## 3 · Open-set verification (equal error rate)

Closed-set re-identification assumes the adversary already holds a list of
candidates. The weaker and more realistic question is verification: given two
clips, are they the same person? We answer it with cosine similarity on the
frozen action features and report the equal error rate, which needs no training.

An EER well below 50% shows the leakage is not an artefact of the closed-set
assumption.

In [ ]:
def verification_eer(ckpt, split='reid_rand', n_pairs=200000, seed=0):
    _, te = SPLITS[split]
    te = np.sort(te)
    F = plain_features(ckpt, te)
    y = np.asarray(SUBJ_LABEL)[te]
    F = F / (np.linalg.norm(F, axis=1, keepdims=True) + 1e-9)

    rng = np.random.RandomState(seed)
    i = rng.randint(0, len(y), n_pairs)
    j = rng.randint(0, len(y), n_pairs)
    keep = i != j
    i, j = i[keep], j[keep]
    s = (F[i] * F[j]).sum(1)
    same = (y[i] == y[j])

    th = np.unique(np.quantile(s, np.linspace(0, 1, 2000)))
    far = np.array([(s[~same] >= t).mean() for t in th])
    frr = np.array([(s[same] < t).mean() for t in th])
    k = np.argmin(np.abs(far - frr))
    return {'eer': 100 * 0.5 * (far[k] + frr[k]), 'threshold': float(th[k]),
            'n_same': int(same.sum()), 'n_diff': int((~same).sum())}


if 'verification' not in REV:
    REV['verification'] = verification_eer(RESULTS['A_action_xsub']['ckpt'])
    save_rev()
v = REV['verification']
print('open-set verification on frozen action features')
print('  EER %.2f%%   (chance 50.00%%)   pairs: %d same / %d different'
      % (v['eer'], v['n_same'], v['n_diff']))

## 4 · Is identity just bone length?

Our preprocessing removes position and orientation but keeps limb proportions,
which are an obvious anthropometric cue and the most likely explanation of the
suppression floor in Section 7.7. This section rebuilds every skeleton with the
population-mean bone lengths, keeping only joint *directions*, then retrains the
action model and re-runs the ladder.

- If leakage falls sharply, identity was largely anthropometric, and a cheap
  input-level normalisation outperforms the adversarial defence. That is a
  strong practical finding and a natural bridge to the next paper.
- If leakage barely moves, identity lives in movement dynamics, and the
  defence problem is harder than bone geometry.

In [ ]:
# bone tree: child -> parent, 0-based, rooted at joint 0 (base of spine)
PARENT = {c: p for c, p in INWARD}
ROOT = [v for v in range(NUM_NODE) if v not in PARENT][0]   # joint 21, the spine hub
CHILDREN = {}
for c, p in INWARD:
    CHILDREN.setdefault(p, []).append(c)

ORDER, seen, frontier = [], {ROOT}, [ROOT]
while frontier:
    nxt = []
    for p in frontier:
        for c in CHILDREN.get(p, []):
            if c not in seen:
                seen.add(c); ORDER.append((c, p)); nxt.append(c)
    frontier = nxt
assert len(ORDER) == NUM_NODE - 1, (len(ORDER), NUM_NODE - 1)
print('bone tree rooted at joint %d; %d bones in traversal order' % (ROOT + 1, len(ORDER)))


def normalise_bone_lengths(Xin, chunk=4000):
    """Rebuild each skeleton with population-mean bone lengths."""
    assert Xin.shape[-1] == 3 and Xin.shape[-2] == NUM_NODE, Xin.shape
    # pass 1: mean length per bone over non-empty frames
    tot = np.zeros(NUM_NODE - 1); cnt = np.zeros(NUM_NODE - 1)
    for a in range(0, len(Xin), chunk):
        blk = np.asarray(Xin[a:a + chunk], dtype=np.float32)
        valid = np.abs(blk).sum(axis=(-1, -2)) > 1e-6          # (..., ) per frame
        for b, (c, p) in enumerate(ORDER):
            d = np.linalg.norm(blk[..., c, :] - blk[..., p, :], axis=-1)
            tot[b] += d[valid].sum(); cnt[b] += valid.sum()
    mean_len = tot / np.maximum(cnt, 1)
    print('mean bone lengths: min %.4f  max %.4f' % (mean_len.min(), mean_len.max()))

    out = np.zeros(Xin.shape, dtype=np.float16)
    for a in range(0, len(Xin), chunk):
        blk = np.asarray(Xin[a:a + chunk], dtype=np.float32)
        new = blk.copy()
        for b, (c, p) in enumerate(ORDER):
            v = blk[..., c, :] - blk[..., p, :]
            n = np.linalg.norm(v, axis=-1, keepdims=True)
            u = np.where(n > 1e-6, v / np.maximum(n, 1e-6), 0.0)
            new[..., c, :] = new[..., p, :] + u * mean_len[b]
        out[a:a + chunk] = new.astype(np.float16)
        if a % (chunk * 5) == 0:
            print('  %d / %d' % (a, len(Xin)), end='\r')
    return out


BN_PATH = '/content/ntu60_T64_bonenorm.npy'
if os.path.exists(BN_PATH):
    X_BN = np.load(BN_PATH, mmap_mode='r')
    print('loaded cached bone-normalised array', X_BN.shape)
else:
    X_BN = normalise_bone_lengths(X)
    np.save(BN_PATH, X_BN)
    X_BN = np.load(BN_PATH, mmap_mode='r')
    print('\nwritten', BN_PATH, X_BN.shape)

In [ ]:
# train the action model and the from-scratch re-ID attacker on normalised input
X_ORIG = X
try:
    X = X_BN
    if 'A_action_bonenorm' not in REV:
        REV['A_action_bonenorm'] = run_experiment(
            tag='A_action_bonenorm', labels=ACT_LABEL, split='xsub',
            num_classes=60, epochs=60, lr=0.1, bs=64)
        save_rev()
    if 'B_reid_bonenorm' not in REV:
        REV['B_reid_bonenorm'] = run_experiment(
            tag='B_reid_bonenorm', labels=SUBJ_LABEL, split='reid_rand',
            num_classes=40, epochs=80, lr=0.1, bs=64)
        save_rev()
    if 'ladder_bonenorm' not in REV:
        REV['ladder_bonenorm'] = ladder_probes(REV['A_action_bonenorm']['ckpt'])
        save_rev()
finally:
    X = X_ORIG        # always restore

print('\n--- bone-length normalised input ---')
print('action (X-Sub)        %.2f%%   (original 86.85%%)' % REV['A_action_bonenorm']['best_acc'])
print('re-ID from scratch    %.2f%%   (original 78.86%%)' % REV['B_reid_bonenorm']['best_acc'])
print('R2 linear probe       %.2f%%   (original 18.67%%)' % REV['ladder_bonenorm']['linear'])
print('R3 MLP probe          %.2f%%   (original 31.46%%)' % REV['ladder_bonenorm']['mlp'])

## 5 · A second backbone — adaptive graph convolution

The reviewer question is whether the findings are specific to vanilla ST-GCN.
This section swaps the fixed adjacency for a 2s-AGCN-style adaptive one: a
learnable residual added to the physical graph, plus a data-dependent attention
term. Everything else — blocks, widths, training recipe — is unchanged, so the
comparison isolates the graph.

We then run the full ladder on it, including the fine-tuning attacker. If the
pattern holds on a stronger backbone, the conclusion generalises beyond ST-GCN.

In [ ]:
class AdaptiveGC(nn.Module):
    """A_k (fixed) + B_k (learnable) + alpha * C_k (data-dependent)."""
    def __init__(self, in_c, out_c, A, inter=None):
        super().__init__()
        self.K = A.shape[0]
        self.register_buffer('A', torch.from_numpy(A.copy()))
        self.B = nn.Parameter(torch.zeros(self.K, A.shape[1], A.shape[2]))
        inter = inter or max(out_c // 4, 16)
        self.theta = nn.ModuleList([nn.Conv2d(in_c, inter, 1) for _ in range(self.K)])
        self.phi = nn.ModuleList([nn.Conv2d(in_c, inter, 1) for _ in range(self.K)])
        self.conv = nn.ModuleList([nn.Conv2d(in_c, out_c, 1) for _ in range(self.K)])
        self.alpha = nn.Parameter(torch.zeros(1))

    def forward(self, x):
        N, _, T, V = x.shape
        y = 0
        for k in range(self.K):
            th = self.theta[k](x).permute(0, 3, 1, 2).reshape(N, V, -1)
            ph = self.phi[k](x).permute(0, 1, 2, 3).reshape(N, -1, V)
            Ck = torch.softmax(torch.bmm(th, ph) / (th.shape[-1] ** 0.5), dim=-1)
            Ak = (self.A[k] + self.B[k]).to(x.dtype)
            z = self.conv[k](x)
            y = y + torch.einsum('nctv,vw->nctw', z, Ak) \
                  + self.alpha.to(x.dtype) * torch.einsum('nctv,nvw->nctw', z, Ck.to(x.dtype))
        return y.contiguous()


def make_agcn(num_class):
    m = STGCN(num_class)
    A = build_adjacency()
    n = 0
    for mod in m.modules():
        if isinstance(mod, STGCNBlock):
            old = mod.gcn
            in_c = old.conv.in_channels
            out_c = old.conv.out_channels // old.K
            mod.gcn = AdaptiveGC(in_c, out_c, A)
            n += 1
    print('replaced %d spatial graph convolutions with adaptive versions' % n)
    return m


# sanity check before committing to a long run
_t = make_agcn(60)
with torch.no_grad():
    _o = _t(torch.zeros(2, IN_CH, T_LEN, NUM_NODE, MAX_PERSON))
    _f = _t.extract(torch.zeros(2, IN_CH, T_LEN, NUM_NODE, MAX_PERSON))
print('forward OK:', tuple(_o.shape), ' features:', tuple(_f.shape))
print('parameters: %.2f M' % (sum(p.numel() for p in _t.parameters()) / 1e6))
del _t

In [ ]:
# train the adaptive-graph backbone by temporarily swapping the model factory
_STGCN_ORIG = STGCN
try:
    STGCN = make_agcn                      # run_experiment builds STGCN(num_classes)
    if 'A_action_agcn' not in REV:
        REV['A_action_agcn'] = run_experiment(
            tag='A_action_agcn', labels=ACT_LABEL, split='xsub',
            num_classes=60, epochs=60, lr=0.1, bs=64)
        save_rev()
    if 'B_reid_agcn' not in REV:
        REV['B_reid_agcn'] = run_experiment(
            tag='B_reid_agcn', labels=SUBJ_LABEL, split='reid_rand',
            num_classes=40, epochs=80, lr=0.1, bs=64)
        save_rev()
    if 'ladder_agcn' not in REV:
        REV['ladder_agcn'] = ladder_probes(REV['A_action_agcn']['ckpt'])
        save_rev()
    if 'ft_agcn' not in REV:
        REV['ft_agcn'] = run_experiment(
            tag='D_finetune_agcn', labels=SUBJ_LABEL, split='reid_rand',
            num_classes=40, epochs=80, lr=0.05, bs=64,
            init_from=REV['A_action_agcn']['ckpt'], freeze=False)
        save_rev()
finally:
    STGCN = _STGCN_ORIG                    # always restore

print('\n--- adaptive-graph backbone ---')
print('action (X-Sub)      %.2f%%   (ST-GCN 86.85%%)' % REV['A_action_agcn']['best_acc'])
print('re-ID from scratch  %.2f%%   (ST-GCN 78.86%%)' % REV['B_reid_agcn']['best_acc'])
print('R2 linear probe     %.2f%%   (ST-GCN 18.67%%)' % REV['ladder_agcn']['linear'])
print('R3 MLP probe        %.2f%%   (ST-GCN 31.46%%)' % REV['ladder_agcn']['mlp'])
print('R4 fine-tune        %.2f%%   (ST-GCN 79.89%%)' % REV['ft_agcn']['best_acc'])

## 6 · Revision summary

Writes `results/revision/revision_tables.txt`. Send me that file plus
`revision.json`.

In [ ]:
L = []
g = lambda d, k, f='best_acc': (REV[d][k] if k in REV.get(d, {}) else None) if f is None \
    else REV.get(d, {}).get(f)

L += ['Table R1. Split protocol and leakage (re-ID from scratch, 40 identities, chance 2.50%)',
      '',
      '%-34s %9s %9s' % ('protocol', 'top-1', 'x chance'),
      '-' * 56,
      '%-34s %8.2f%% %8.1fx' % ('random 80/20 (original)', 78.86, 31.5),
      '%-34s %8.2f%% %8.1fx' % ('cameras 1,2 -> 3 (original)', 56.26, 22.5)]
for tag, lab in [('B_reid_take', 'take-grouped (no shared takes)'),
                 ('B_reid_xsetup', 'cross-setup (even -> odd)')]:
    if tag in REV:
        a = REV[tag]['best_acc']
        L.append('%-34s %8.2f%% %8.1fx' % (lab, a, a / 2.5))

if 'ladder_seeds' in REV and REV['ladder_seeds']:
    L += ['', 'Table R2. Ladder rungs over backbone seeds (random split)', '']
    for fld, lab in [('linear', 'R2 linear probe'), ('mlp', 'R3 MLP probe')]:
        v = np.array([r[fld] for r in REV['ladder_seeds'].values()])
        L.append('%-24s %.2f +/- %.2f   (n=%d)' % (lab, v.mean(), v.std(ddof=1), len(v)))

if 'verification' in REV:
    L += ['', 'Table R3. Open-set verification on frozen action features', '',
          '  equal error rate  %.2f%%   (chance 50.00%%)' % REV['verification']['eer']]

L += ['', 'Table R4. Input and backbone variants (random split)', '',
      '%-26s %9s %9s %9s %9s %9s' % ('variant', 'action', 'R5 scratch', 'R2', 'R3', 'R4'),
      '-' * 78,
      '%-26s %8.2f%% %8.2f%% %8.2f%% %8.2f%% %8.2f%%'
      % ('ST-GCN, joints (paper)', 86.85, 78.86, 18.67, 31.46, 79.89)]
if 'A_action_bonenorm' in REV:
    L.append('%-26s %8.2f%% %8.2f%% %8.2f%% %8.2f%% %9s'
             % ('ST-GCN, bone-normalised', REV['A_action_bonenorm']['best_acc'],
                REV['B_reid_bonenorm']['best_acc'], REV['ladder_bonenorm']['linear'],
                REV['ladder_bonenorm']['mlp'], '-'))
if 'A_action_agcn' in REV:
    L.append('%-26s %8.2f%% %8.2f%% %8.2f%% %8.2f%% %8.2f%%'
             % ('adaptive graph, joints', REV['A_action_agcn']['best_acc'],
                REV['B_reid_agcn']['best_acc'], REV['ladder_agcn']['linear'],
                REV['ladder_agcn']['mlp'], REV['ft_agcn']['best_acc']))

txt = '\n'.join(L)
print(txt)
open(os.path.join(REV_DIR, 'revision_tables.txt'), 'w').write(txt)
save_rev()
print('\nwritten to', REV_DIR)